# Puzzletree demo

This notebook builds a shuffled tile set from the bundled `city.jpg` image, runs the reconstruction pipeline, and previews the shuffled and reconstructed images.

In [ ]:
from pathlib import Path
import random
import shutil

from IPython.display import display
from PIL import Image

from puzzletree.reconstruct import ReconstructOptions, run_from_options
from puzzletree.reconstruct.io import split_image_into_tiles

repo_root = Path.cwd()
if not (repo_root / "tests" / "test_data").exists():
    repo_root = repo_root.parent
if not (repo_root / "tests" / "test_data").exists():
    raise RuntimeError("Run this notebook from the repository root or the notebook directory.")

demo_dir = repo_root / "notebook" / "demo_output"
tiles_dir = demo_dir / "tiles"
reconstructed_path = demo_dir / "reconstructed.png"
source_image_path = repo_root / "tests" / "test_data" / "city.jpg"

if demo_dir.exists():
    shutil.rmtree(demo_dir)
tiles_dir.mkdir(parents=True)

source_image = Image.open(source_image_path).convert("RGB")
tiles = split_image_into_tiles(source_image, rows=4, cols=5)
order = list(range(len(tiles)))
random.Random(7).shuffle(order)

tile_w, tile_h = tiles[0].size
shuffled_preview = Image.new("RGB", source_image.size)
for output_index, tile_index in enumerate(order):
    row, col = divmod(output_index, 5)
    tile = tiles[tile_index]
    tile.save(tiles_dir / f"tile_{output_index:03d}.png")
    shuffled_preview.paste(tile, (col * tile_w, row * tile_h))

result = run_from_options(
    ReconstructOptions(
        input_dir=tiles_dir,
        output=reconstructed_path,
    )
)

print(f"Wrote {len(order)} shuffled tiles to {tiles_dir.relative_to(repo_root)}")
print(f"Saved reconstructed image to {reconstructed_path.relative_to(repo_root)}")
print(f"Placed {len(result.placements)} tiles across {len(result.adjs)} adjacency entries")


## Shuffled input vs reconstructed output

In [ ]:
display(shuffled_preview)
display(result.output)